# MoodMirror M4 Ablations (Colab Runner)

This notebook runs all M4 ablation configs on GPU and aggregates results.

Expected project layout after upload:
- m4/models.py
- m4/data/data_loader.py
- m4/experiments/run_exp.py
- m4/experiments/experiment_logger.py
- m4/experiments/aggregate_results.py
- m4/experiments/utils.py
- m4/configs/*.yaml

Dataset location (uploaded manually to Drive):
- /content/data/fer2013/train/<class folders>
- /content/data/fer2013/test/<class folders>

In [ ]:
# Folder created after unzipping for_ablation.zip in Colab
PROJECT_ROOT = '/content/for_ablation'
DATA_PATH = '/content/data/fer2013'

import os
if not os.path.isdir(PROJECT_ROOT):
    raise FileNotFoundError(f'PROJECT_ROOT not found: {PROJECT_ROOT}')
if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(f'DATA_PATH not found: {DATA_PATH}')

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q pyyaml tensorboard scikit-learn pandas matplotlib seaborn

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import glob
import os
import yaml

required_paths = [
    'm4/models.py',
    'm4/data/data_loader.py',
    'm4/experiments/run_exp.py',
    'm4/experiments/experiment_logger.py',
    'm4/experiments/aggregate_results.py',
    'm4/experiments/utils.py',
    'm4/configs',
]

missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))
if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(f'Dataset path not found: {DATA_PATH}')

configs = sorted(glob.glob('m4/configs/*.yaml'))
print('Found configs:', len(configs))
for c in configs:
    print(' -', c)

# Force all configs to use the Colab root dataset path
for cfg_path in configs:
    with open(cfg_path, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['data_path'] = DATA_PATH
    with open(cfg_path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

train_count = sum(1 for _ in glob.iglob(os.path.join(DATA_PATH, 'train', '*', '*')))
test_count = sum(1 for _ in glob.iglob(os.path.join(DATA_PATH, 'test', '*', '*')))
print('Train images:', train_count)
print('Test images: ', test_count)

In [ ]:
import os
import sys
import time
import glob
import subprocess

configs = sorted(glob.glob('m4/configs/*.yaml'))
os.makedirs('m4/logs', exist_ok=True)

start_all = time.time()
failures = []

for idx, cfg_path in enumerate(configs, 1):
    cfg_name = os.path.basename(cfg_path)
    print(f'\n[{idx}/{len(configs)}] Running {cfg_name}')
    t0 = time.time()

    log_path = os.path.join('m4', 'logs', cfg_name.replace('.yaml', '.log'))
    cmd = [sys.executable, 'run_exp.py', '--config', f'../configs/{cfg_name}']

    with open(log_path, 'w') as log_file:
        proc = subprocess.run(
            cmd,
            cwd='m4/experiments',
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True
        )

    dt = time.time() - t0
    print(f'Finished {cfg_name} in {dt/60:.1f} min with code {proc.returncode}')

    if proc.returncode != 0:
        failures.append((cfg_name, log_path))

total_minutes = (time.time() - start_all) / 60
print(f'\nTotal runtime: {total_minutes:.1f} min')

if failures:
    print('\nSome runs failed:')
    for name, log in failures:
        print(' -', name, 'log:', log)
    raise RuntimeError('One or more ablations failed. Check m4/logs/*.log')
else:
    print('\nAll ablation runs finished successfully.')

In [ ]:
import os
import sys
import subprocess

cmd = [sys.executable, 'aggregate_results.py']
proc = subprocess.run(cmd, cwd='m4/experiments', text=True)
if proc.returncode != 0:
    raise RuntimeError('Aggregation failed')

print('Aggregation complete.')
print('Runs directory:', os.path.abspath('m4/runs'))

In [ ]:
import os
import shutil

artifact_base = 'm4_ablation_artifacts'
artifact_zip = shutil.make_archive(artifact_base, 'zip', root_dir='m4')
print('Created:', os.path.abspath(artifact_zip))

from google.colab import files
files.download(artifact_zip)